# ఓపెన్ ఏఐ మోడల్స్‌ను ఫైన్ ట్యూనింగ్ చేయడం

ఈ నోట్‌బుక్ ప్రస్తుతం ఓపెన్ ఏఐ నుండి అందుబాటులో ఉన్న [ఫైన్ ట్యూనింగ్](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst) డాక్యుమెంటేషన్ ఆధారంగా రూపొందించబడింది.

> **గమనిక:** ఈ నోట్‌బుక్ ఉదాహరణ అవుట్పుట్ `gpt-3.5-turbo`పై రూపొందించబడింది, ఇది ఇప్పుడు మైక్రోసాఫ్ట్ ఫౌండ్రీలో ఏజూర్ ఓపెన్ ఏఐలో ఇన్ఫెరెన్స్ మరియు ఫైన్-ట్యూనింగ్ రెండింటికీ విరమింపబడింది (మరియు ఓపెన్ ఏఐనిచే నేరుగా డిప్రికేటెడ్ చేయబడింది). క్రింద ఇచ్చిన వాక్‌థ్రూ మరియు تصورాలు ఇంకా సరైనవి, కానీ మీరు కొత్త ఫైన్-ట్యూనింగ్ పనిని ఈ రోజు మొదలుపెట్టుతున్నట్లయితే ప్రస్తుతం మద్దతు ఇచ్చే మోడల్‌లను లక్ష్యంగా పెట్టండి - ఉదాహరణకు `gpt-4o-mini` లేదా `gpt-4.1-mini`. ప్రస్తుతం ఫైన్-ట్యూన్ చేయదగిన మోడల్స్ సెట్ కోసం [ఫైన్-ట్యూనింగ్ మోడల్స్ జాబితా](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models) చూడండి.

ఫైన్-ట్యూనింగ్ అనేది మీ అనువర్తనానికి సంబంధించిన అదనపు డేటా మరియు సందర్భంతో మోడల్‌ను మళ్లీ శిక్షణ ఇస్తూ ప్రదర్శనను మెరుగుపరుస్తుంది. _few shot learning_ మరియు _retrieval augmented generation_ వంటి ప్రాంప్ట్ ఇంజనీరింగ్ ప్రాజ్ఞలు, ప్రామాణిక ప్రాంప్ట్‌ను సంబంధిత డేటాతో మెరుగుపరుచడానికి అనుమతిస్తాయి. అయితే, ఈ పద్ధతులు లక్ష్య మోడల్ యొక్క గరిష్ట టోకెన్ విండో పరిమితితో పరిమితమవుతాయి.

ఫైన్-ట్యూనింగ్‌తో, అవసరమైన డేటాతో మోడల్‌ను స్వయంగా మళ్లీ శిక్షణ ఇస్తాము (గరిష్ట టోకెన్ విండోలో సరిపోయే కన్నా ఎక్కువ ఉదాహరణలను ఉపయోగించుకోవడం సౌకర్యంగా ఉంటుంది) - మరియు ప్రక్రియ సమయంలో ఉదాహరణలు అవసరం లేని _కస్టమ్_ మోడల్ వెర్షన్‌ను నిష్పత్తి చేస్తాము. ఇది కేవలం మా ప్రాంప్ట్ రూపకల్పన ఫలితాలను మెరుగుపరచటమే కాకుండా (మేము టోకెన్ విండోను ఇతర పనులకు ఉపయోగించుకునే అవకాశం పెరుగుతుంది), ఇన్ఫెరెన్స్ సమయంలో మోడల్‌కు పంపాల్సిన టోకెన్ల సంఖ్య తగ్గడం వలన ఖర్చులను కూడా తగ్గిస్తుంది.

ఫైన్ ట్యూనింగ్‌లో 4 దశలు ఉంటాయి:
1. శిక్షణ డేటాను సిద్ధం చేసి అప్లోడ్ చేయడం.
1. శిక్షణ నોકરીను నడిపించి ఫైన్-ట్యూన్ చేసిన మోడల్ పొందడం.
1. ఫైన్-ట్యూన్ చేసిన మోడల్‌ను మూల్యాంకనం చేసి నాణ్యత కోసం పునరావృతం చేయడం.
1. సంతృప్తిగా ఉన్నప్పుడు ఫైన్-ట్యూన్ చేసిన మోడల్‌ను ఇన్ఫెరెన్స్ కోసం డీ ప్లాయ్ చేయడం.

అన్ని ఫౌండేషన్ మోడల్స్ ఫైన్-ట్యూనింగ్‌కు మద్దతు ఇవ్వవు - తాజా సమాచారం కోసం [OpenAI డాక్యుమెంటేషన్](https://platform.openai.com/docs/guides/fine-tuning/what-models-can-be-fine-tuned?WT.mc_id=academic-105485-koreyst) ను చూడండి. మీరు ఇప్పటికే ఫైన్-ట్యూన్ చేసిన మోడల్‌ను కూడా పునఃఫైన్-ట్యూన్ చేయవచ్చు. ఈ ట్యుటోరియల్‌లో, మేము ఫైన్-ట్యూనింగ్‌ కోసం లక్ష్య ఫౌండేషన్ మోడల్‌గా `gpt-35-turbo` ఉపయోగించబోతున్నాము (పైన ఉన్న గమనికలో ప్రస్తుత మద్దతు ఇచ్చే ప్రత్యామ్నాయాన్ని చూడండి).

---


### దశ 1.1: మీ డేటాసెట్‌ను సిద్ధం చేసుకోండి

మేము ఒక చాట్‌బోట్‌ను తయారు చేద్దాం, ఇది ఒక మూలకం గురించి ప్రశ్నలకు లిమెరిక్ ద్వారా సమాధానాలు చెప్పి పీరియాడిక్ టేబుల్‌ను మీకు అర్థం చేసుకోవడంలో సహాయం చేస్తుంది. ఈ సులభమైన ట్యుటోరియల్‌లో, మేము కేవలం డేటా ఫార్మాట్ ఎలా ఉండాలో చూపించేందుకు కొన్ని ఉదాహరణలతో డేటాసెట్‌ను మాత్రమే రూపొందిస్తాము. వాస్తవ ప్రపంచంలో మీరు շատ ఎక్కువ ఉదాహరణలతో కూడిన డేటాసెట్ అవసరం అవుతుంది. అవసరమైతే మీ అనువర్తన డొమైన్కు సంబంధించిన ఓపెన్ డేటాసెట్‌ను కూడా ఉపయోగించి, ఫైన్-ట్యూనింగ్ కోసం దాన్ని రీ ఫార్మట్ చేయవచ్చు.

మేము `gpt-35-turbo`పై కేంద్రీకృతమై, ఒకటి మాట లావాదేవీ (చాట్ కంప్లీషన్) కోసం ఉదాహరణలు సృష్టిస్తున్నందున, [ఈ సూచించిన ఫార్మాట్](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst) ను ఉపయోగించి OpenAI చాట్ కంప్లీషన్ అవసరాల ప్రకారం ఉదాహరణలు సృష్టించవచ్చు. మీరు బహుళ-లావాదేవీ సంభాషణ కంటెంట్ ఆశిస్తున్నట్లయితే, [బహుళ-లావాదేవీ ఉదాహరణ ఫార్మాట్](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst) ఉపయోగించాలి; ఇందులో `weight` పారామీటర్ ఉంటుంది, ఇది ఫైన్-ట్యూనింగ్ ప్రక్రియలో ఏ సందేశాలు ఉపయోగించాలో సూచిస్తుంది.

మా ట్యుటోరియల్ కోసం మేము సులభమైన ఒకటి మాట ఫార్మాట్‌ను ఉపయోగిస్తాము. డేటా [jsonl ఫార్మాట్](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst) లో ఉంటుంది, ప్రతి లైన్‌లో ఒక రికార్డు ఉంటుంది, ప్రతి ఒక్కటి JSON ఫార్మాటెడ్ ఆబ్జెక్ట్‌గా ఉంటుంది. క్రింది కోడ్‌లో 2 రికార్డులు నమూనాగా చూపబడ్డాయి - పూర్తి 10 ఉదాహరణల నమూనా సె트를 [training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl) లో చూడండి, ఇది మా ఫైన్-ట్యూనింగ్ ట్యుటోరియల్ కోసం ఉపయోగిస్తాము. **గమనిక:** ప్రతి రికార్డు _ఒక్కదfassungన లైన్‌లో ఉండాలి_ (సాధారణ JSON ఫైల్స్ విధంగా పంక్తులుగా విడగొట్టకుండా)

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

వాస్తవ ప్రపంచంలో మంచి ఫలితాల కోసం మీరు చాలా పెద్ద ఉదాహరణల సెట్ అవసరం అవుతుంది - ఫలితాల నాణ్యత మరియు ఫైన్-ట్యూనింగ్‌కు కావలసిన సమయం/ఖర్చుల మధ్య ఒక పరివర్తన ఉంటుంది. మేము చిన్న సెట్ ఉపయోగిస్తున్నాము, అంటే ఫైన్-ట్యూనింగ్ త్వరగా పూర్తి చేసి ప్రక్రియను చూపించగలుగుతాం. మరింత సంక్లిష్టమైన ఫైన్-ట్యూనింగ్ ట్యుటోరియల్ కోసం [ఈ OpenAI కుక్బుక్ ఉదాహరణ](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) చూడండి.


---

### దశ 1.2 మీ డేటాసెట్‌ను అప్‌లోడ్ చేయండి

ఫైల్స్ API ఉపయోగించి డేటాను అప్‌లోడ్ చేయండి [ఇక్కడ వివరించినట్టు](https://platform.openai.com/docs/guides/fine-tuning/upload-a-training-file). ఈ కోడ్‌ను నడిపించడానికి, మీరు ముందుగా ఈ క్రింది దశలను పూర్తి చేసి ఉండాలి:
 - `openai` పైథాన్ ప్యాకేజీని ఇన్‌స్టాల్ చేసుకున్నారు (తాజా ఫీచర్ల కోసం >=0.28.0 వర్షన్ ఉపయోగించండి)
 - `OPENAI_API_KEY` అనే ఎన్విరాన్‌మెంట్ వేరియబుల్‌ను మీ OpenAI API కీతో సెట్ చేసుకున్నారు
మరింత తెలుసుకోవడానికి, కోర్సు కోసం అందించిన [సెట్టప్ గైడ్](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst) ను చూడండి.

ఇప్పుడు, మీ స్థానిక JSONL ఫైల్ నుండి అప్‌లోడ్ కోసం ఫైల్ సృష్టించే కోడ్ నడపండి.


In [24]:
from openai import OpenAI
client = OpenAI()

ft_file = client.files.create(
  file=open("./training-data.jsonl", "rb"),
  purpose="fine-tune"
)

print(ft_file)
print("Training File ID: " + ft_file.id)

FileObject(id='file-JdAJcagdOTG6ACNlFWzuzmyV', bytes=4021, created_at=1715566183, filename='training-data.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None)
Training File ID: file-JdAJcagdOTG6ACNlFWzuzmyV


---

### దశ 2.1: SDK తో ఫైన్-ట్యూనింగ్ జాబ్ సృష్టించండి


In [25]:
from openai import OpenAI
client = OpenAI()

ft_filejob = client.fine_tuning.jobs.create(
  training_file=ft_file.id, 
  model="gpt-3.5-turbo"
)

print(ft_filejob)
print("Fine-tuning Job ID: " + ft_filejob.id)

FineTuningJob(id='ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', created_at=1715566184, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(n_epochs='auto', batch_size='auto', learning_rate_multiplier='auto'), model='gpt-3.5-turbo-0125', object='fine_tuning.job', organization_id='org-EZ6ag0n0S6Zm8eV9BSWKmE6l', result_files=[], seed=830529052, status='validating_files', trained_tokens=None, training_file='file-JdAJcagdOTG6ACNlFWzuzmyV', validation_file=None, estimated_finish=None, integrations=[], user_provided_suffix=None)
Fine-tuning Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh


---

### దశ 2.2: జాబ్ స్థితిని తనిఖీ చేయండి

ఇక్కడ `client.fine_tuning.jobs` API తో మీరు చేయగల కొన్ని విషయాలు ఉన్నాయి:
- `client.fine_tuning.jobs.list(limit=<n>)` - చివరి n ఫైన్-ట్యూనింగ్ జాబ్స్ ను జాబితా చేయండి
- `client.fine_tuning.jobs.retrieve(<job_id>)` - ఒక నిర్దిష్ట ఫైన్-ట్యూనింగ్ జాబ్ వివరాలను పొందండి
- `client.fine_tuning.jobs.cancel(<job_id>)` - ఒక ఫైన్-ట్యూనింగ్ జాబ్ ను రద్దు చేయండి
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<b>)` - జాబ్ నుండి n ఈవెంట్లను జాబితా చేయండి
- `client.fine_tuning.jobs.create(model="gpt-35-turbo", training_file="your-training-file.jsonl", ...)`

ప్రోసెస్ యొక్క మొదటి దశ _ట్రైనింగ్ ఫైల్ ని నిర్ధారించడం_; ఇది డేటా సరైన ఫార్మాట్ లో ఉన్నదో లేదో చూసుకోవడం.


In [26]:
from openai import OpenAI
client = OpenAI()

# List 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the state of a fine-tune
client.fine_tuning.jobs.retrieve(ft_filejob.id)

# List up to 10 events from a fine-tuning job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_filejob.id, limit=10)

SyncCursorPage[FineTuningJobEvent](data=[FineTuningJobEvent(id='ftevent-GkWiDgZmOsuv4q5cSTEGscY6', created_at=1715566184, level='info', message='Validating training file: file-JdAJcagdOTG6ACNlFWzuzmyV', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-3899xdVTO3LN7Q7LkKLMJUnb', created_at=1715566184, level='info', message='Created fine-tuning job: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh', object='fine_tuning.job.event', data={}, type='message')], object='list', has_more=False)

In [30]:
# Once the training data is validated
# Track the job status to see if it is running and when it is complete
from openai import OpenAI
client = OpenAI()

response = client.fine_tuning.jobs.retrieve(ft_filejob.id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)

Job ID: ftjob-Usfb9RjasncaZ5Cjbuh1XSCh
Status: running
Trained Tokens: None


---

### దశ 2.3: పురోగతిని గమనించడానికి ఈవెంట్లను ట్రాక్ చేయండి


In [44]:
# You can also track progress in a more granular way by checking for events
# Refresh this code till you get the `The job has successfully completed` message
response = client.fine_tuning.jobs.list_events(ft_filejob.id)

events = response.data
events.reverse()

for event in events:
    print(event.message)

Step 85/100: training loss=0.14
Step 86/100: training loss=0.00
Step 87/100: training loss=0.00
Step 88/100: training loss=0.07
Step 89/100: training loss=0.00
Step 90/100: training loss=0.00
Step 91/100: training loss=0.00
Step 92/100: training loss=0.00
Step 93/100: training loss=0.00
Step 94/100: training loss=0.00
Step 95/100: training loss=0.08
Step 96/100: training loss=0.05
Step 97/100: training loss=0.00
Step 98/100: training loss=0.00
Step 99/100: training loss=0.00
Step 100/100: training loss=0.00
Checkpoint created at step 80 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyyF2:ckpt-step-80
Checkpoint created at step 90 with Snapshot ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWyzhK:ckpt-step-90
New fine-tuned model created: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz
The job has successfully completed


### దశ 2.4: OpenAI డ్యాష్‌బోర్డ్‌లో స్థితి చూడండి


మీరు OpenAI వెబ్సైట్‌ను సందర్శించి ప్లాట్‌ఫామ్ యొక్క _ఫైన్-ట్యూనింగ్_ విభాగాన్ని అన్వేషించడం ద్వారా కూడా స్థితిని చూడవచ్చు. ఇది ప్రస్తుత పని స్థితిని మీకు చూపిస్తుంది, అలాగే పూర్వ పని అమలు పరుగుల చరిత్రను ట్రాక్ చేయడానికి కూడా అనుమతిస్తుంది. ఈ స్క్రీన్‌షాట్‌లో, మీరు పూర్వ అమలు విఫలమైంది, మరియు రెండవ రన్ విజయవంతమైంది అని చూడవచ్చు. సందర్భానికి, ఇది మొదటి రన్ తప్పుగా ఫార్మాట్ చేసిన రికార్డులతో ఉన్న JSON ఫైల్‌ను ఉపయోగించినప్పుడు జరిగింది - ఒకసారి సరి చేయబడిన తరువాత, రెండవ రన్ విజయవంతంగా పూర్తి ఐ, మరియు మోడల్‌ను ఉపయోగానికి అందుబాటులో ఉంచింది.

![Fine-tuning job status](../../../../../translated_images/te/fine-tuned-model-status.563271727bf7bfba.webp)


మీరు క్రిందకి స్క్రోల్ చేసి దృశ్య డాష్‌బోర్డులో స్టేటస్ సందేశాలు మరియు మేరిక్స్ కూడా చూడవచ్చు, క్రింద చూపినట్లు:

| Messages | Metrics |
|:---|:---|
| ![Messages](../../../../../translated_images/te/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/te/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### దశ 3.1: ID పొందండి & కోడులో ఫైన్-ట్యూన్ చేయబడిన మోడల్‌ను పరీక్షించండి


In [46]:
# Retrieve the identity of the fine-tuned model once ready
response = client.fine_tuning.jobs.retrieve(ft_filejob.id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)

Fine-tuned Model ID: ft:gpt-3.5-turbo-0125:bitnbot::9OFWzNjz


In [ ]:
# You can then use that model to generate completions from the SDK as shown
# Or you can load that model into the OpenAI Playground (in the UI) to validate it from there.
from openai import OpenAI
client = OpenAI()

completion = client.responses.create(
  model=fine_tuned_model_id,
  input=[
    {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
    {"role": "user", "content": "Tell me about Strontium"},
  ],
  store=False,
)
print(completion.output_text)


ChatCompletionMessage(content="Strontium, a metal so bright - It's in fireworks, a dazzling sight - It's in bones, you see - And in tea, it's the key - It's the fortieth, so pure, that's the right", role='assistant', function_call=None, tool_calls=None)


---

### దశ 3.2: ప్లేగ్రౌండ్‌లో ఫైన్-ట్యూన్ చేసిన మోడల్‌ను లోడ్ చేయండి & పరీక్షించండి

మీరు ఇప్పుడు రెండు రకాల్లో ఫైన్-ట్యూన్ అయిన మోడల్‌ని పరీక్షించవచ్చు. మొదటగా, మీరు ప్లేగ్రౌండ్‌ను సందర్శించి మోడల్స్ డ్రాప్-డౌన్ నుండి మీ కొత్తగా ఫైన్-ట్యూన్ చేసిన మోడల్‌ను ఎంపిక చేసుకోవచ్చు. మరొక ఎంపిక "ఫైన్-ట్యూనింగ్" ప్యానెల్లో చూపబడిన "ప్లేగ్రౌండ్" ఎంపికను ఉపయోగించడం, ఇది క్రింది _comparaaltive_ వీక్షణను ప్రారంభిస్తుంది, ఇది ఫౌండేషన్ మరియు ఫైన్-ట్యూన్ చేసిన మోడల్ వర్షన్లను పక్కపక్కగా చూపిస్తుంది వేగమైన మూల్యాంకనానికి.

![ఫైన్-ట్యూనింగ్ పని స్థితి](../../../../../translated_images/te/fine-tuned-playground-compare.56e06f0ad8922016.webp)

మీరు మీ శిక్షణ డేటాలో ఉపయోగించిన సిస్టమ్ కాంటెక్స్ట్‌ను సింపులీ భర్తీ చేసి, మీ పరీక్ష ప్రశ్నను అందించండి. మీరు గమనిస్తారు రెండు వైపులా సమానమైన కాంటెక్స్ట్ మరియు ప్రశ్నతో అప్డేట్ అవుతాయి. పోలికను నడిపించండి, అవుట్పుట్లలో తేడా చూడగలుగుతారు. _ఫైన్-ట్యూన్ చేసిన మోడల్ మీ నమూనాల్లో మీరు ఇచ్చిన ఫార్మాట్ లోనే ప్రతిస్పందన చేస్తుంది, కానీ ఫౌండేషన్ మోడల్ సిస్టమ్ ప్రాంప్ట్‌ను అనుసరిస్తుంది_.

![ఫైన్-ట్యూనింగ్ పని స్థితి](../../../../../translated_images/te/fine-tuned-playground-launch.5a26495c983c6350.webp)

మీరు గమనిస్తారు పోలిక ఇరువురు మోడల్స్ కోసం టోకెన్ లెక్కలు మరియు ఇన్ఫెరెన్స్కు తీసుకున్న సమయాన్ని కూడా అందిస్తుంది. **ఈ ప్రత్యేక ఉదాహరణ ప్రాసెస్ చూపించడానికి సరళమైనదిగా ఉంది కానీ నిజ జీవిత డేటాసెట్ లేదా పరిస్థితికి ఇతరమయిన ఉదాహరణ కాదు**. మీరు గమనించవచ్చు రెండూ ఉదాహరణలు సమాన టోకెన్ సంఖ్యను చూపుతున్నాయి (సిస్టమ్ కాంటెక్స్ట్ మరియు వినియోగదారు ప్రాంప్ట్ సమానమే) అయితే ఫైన్-ట్యూన్ చేసిన మోడల్ ఇన్ఫెరెన్స్ కోసం ఎక్కువ సేపు తీసుకుంటుంది (కస్టమ్ మోడల్).

నిజ జీవిత పరిస్థితులలో, మీరు ఇలాంటి ఆటపాట ఉదాహరణ ఉపయోగించడం కాకుండా నిజమైన డేటా మీద ఫైన్-ట్యూనింగ్ చేస్తారు (ఉదా: కస్టమర్ సర్వీసు కోసం ఉత్పత్తి కేటలాగ్) అక్కడ స్పందన యొక్క నాణ్యత మరింత స్పష్టంగా కనిపిస్తుంది. _ఆ సందర్భంలో_, సమానమైన స్పందన నాణ్యతను ఫౌండేషన్ మోడల్‌తో పొందడానికి మరింత అనుకూల ప్రాంప్ట్ ఇంజనీరింగ్ చేయాలి, ఇది టోకెన్ ఉపయోగాన్ని మరియు ఇన్ఫెరెన్స్ కోసం సంబంధించిన ప్రాసెసింగ్ సమయాన్ని పెంచుతుంది. _దీనిని ప్రయత్నించడానికి, OpenAI కుక్బుక్‌లోని ఫైన్-ట్యూనింగ్ ఉదాహరణలను చూడండి._

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
